In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import time
import unyt as u

import richio
import dev

## Nozzle shock check vs Ryu+ 2025

Back-of-envelope check against [arXiv:2510.04790](https://arxiv.org/abs/2510.04790), which finds
the nozzle-shock dissipation to be ~4e-5 of the orbital energy (their order-of-magnitude estimate
is `dE/E ~ (v_th/v_bulk)^2 ~ 1e-4`, converging to ~4e-5 at their highest resolution). Their actual
measured quantity is the **ratio of thermal (internal) energy to kinetic energy** near pericenter.

Our sim (`R0.47M0.5BH10000beta1S60n1.5ComptonHiResNewAMR`) is a full (beta=1) disruption too, but
Mbh=1e4 Msun instead of their 1e6 Msun.

Matching their setup:

- **Frame.** The first ~21 snapshots are in the star's comoving frame; from snap_22 on the sim is
  in the **global BH frame**, so the snapshot velocity field is the true orbital velocity. We only
  measure in the BH frame.
- **Time.** One fallback time is `t_fb = 2.577726 day`, so `t/t_fb = tfb[day] / 2.577726`. We
  evaluate around **0.3 t_fb**.
- **Selection.** Like the paper, divide the orbital plane (xy here) into 3-deg angular beams, pick
  the beam with peak kinetic energy, and take it +/- 4.5 deg (3 beams).

We compute two things in that selection:

1. **Internal/kinetic** `sum(IE*mass) / sum(KE)` -> the paper's actual quantity (accumulated
   thermal energy vs kinetic).
2. **Rate construction** `(sum(Ediss_rate*Volume) / sum(KE)) * (L / v_peri)` -> the instantaneous
   dissipation *rate* turned into an energy over one pericenter crossing, with `L = Rp` and
   `v_peri = sqrt(2 G Mbh / Rp)`.

In [2]:
base = "/data1/projects/pi-rossiem/TDE_data/R0.47M0.5BH10000beta1S60n1.5ComptonHiResNewAMR"

mstar = 0.5 * richio.units.mscale
rstar = 0.47 * richio.units.lscale
mbh = 1e4 * richio.units.mscale
G = u.physical_constants.G
t_fb = 2.577726 * u.day  # fallback time for this system (richio NPY tfb unit)

rt = rstar * (mbh / mstar) ** (1 / 3)
rp = rt  # beta = 1, full disruption
v_peri = np.sqrt(2 * G * mbh / rp)  # global BH-frame speed at pericenter

peri_length = rp  # "length of pericenter passage" ~ Rp
crossing_time = peri_length / v_peri  # let unyt keep the units; convert only for display

print(f"rp = {rp.to('cm'):.3e}")
print(f"v_peri = {v_peri.to('km/s'):.3e}")
print(f"peri_length = Rp = {peri_length.to('cm'):.3e}, crossing_time = {crossing_time.to('s'):.3e}")
print(f"t_fb = {t_fb}, 0.3 t_fb = {0.3 * t_fb}")

rp = 8.930e+11 cm
v_peri = 1.729e+04 km/s
peri_length = Rp = 8.930e+11 cm, crossing_time = 5.165e+02 s
t_fb = 2.577726 day, 0.3 t_fb = 0.7733178 day


## Find 0.3 t_fb in the BH frame

Scan `t/t_fb` across BH-frame snapshots (snap_22 on). snap_44 lands at t/t_fb = 0.305, so that's
our evaluation snapshot.

In [3]:
for i in range(22, 54, 2):
    snap_i = richio.load(f"{base}/snap_{i}")
    t_day = snap_i.tfb.in_units("day")
    print(f"snap_{i:3d}  t = {float(t_day):.4f} d   t/t_fb = {float(t_day / t_fb):.4f}")

snap_ 22  t = 0.1379 d   t/t_fb = 0.0535
snap_ 24  t = 0.1709 d   t/t_fb = 0.0663
snap_ 26  t = 0.2083 d   t/t_fb = 0.0808
snap_ 28  t = 0.2503 d   t/t_fb = 0.0971
snap_ 30  t = 0.2972 d   t/t_fb = 0.1153
snap_ 32  t = 0.3503 d   t/t_fb = 0.1359
snap_ 34  t = 0.4078 d   t/t_fb = 0.1582
snap_ 36  t = 0.4711 d   t/t_fb = 0.1828
snap_ 38  t = 0.5408 d   t/t_fb = 0.2098
snap_ 40  t = 0.6163 d   t/t_fb = 0.2391
snap_ 42  t = 0.6981 d   t/t_fb = 0.2708
snap_ 44  t = 0.7867 d   t/t_fb = 0.3052
snap_ 46  t = 0.8822 d   t/t_fb = 0.3422
snap_ 48  t = 0.9863 d   t/t_fb = 0.3826
snap_ 50  t = 1.0966 d   t/t_fb = 0.4254
snap_ 52  t = 1.2147 d   t/t_fb = 0.4712


## Dissipated fraction at 0.3 t_fb, paper's angular-beam selection

Build the 3-deg angular beams in the orbital (xy) plane, pick the peak-KE beam +/- one beam, and
report both the internal/kinetic ratio and the dissipation-rate construction there (and, for
reference, over all stellar debris with no angular cut).

In [4]:
def load_star_fields(snap_num):
    snap_i = richio.load(f"{base}/snap_{snap_num}")
    star = snap_i.mask_star_ratio()
    cell_mass = (snap_i.density * snap_i.volume)[star]
    vx, vy, vz = snap_i.velocity_x[star], snap_i.velocity_y[star], snap_i.velocity_z[star]

    kinetic_energy = 0.5 * cell_mass * (vx**2 + vy**2 + vz**2)
    thermal_energy = snap_i.internal_energy[star] * cell_mass  # specific IE * mass
    dissipation_power = snap_i.dissipation[star] * snap_i.volume[star]  # erg/s per cell
    azimuth = np.arctan2(np.asarray(snap_i.CMy[star]), np.asarray(snap_i.CMx[star]))

    t_over_tfb = float(snap_i.tfb / t_fb)  # both times -> unyt cancels to dimensionless
    return t_over_tfb, kinetic_energy, thermal_energy, dissipation_power, azimuth


def peak_kinetic_beam(azimuth, kinetic_energy, n_beams=120):
    """3-deg angular beams in the xy plane; select peak-KE beam +/- one beam (+/-4.5 deg)."""
    beam = (np.floor((azimuth + np.pi) / (2 * np.pi) * n_beams).astype(int)) % n_beams
    energy_per_beam = np.bincount(beam, weights=np.asarray(kinetic_energy), minlength=n_beams)
    peak = int(np.argmax(energy_per_beam))
    return np.isin(beam, [(peak - 1) % n_beams, peak, (peak + 1) % n_beams])


def report(tag, kinetic_energy, thermal_energy, dissipation_power, mask):
    kinetic = np.sum(kinetic_energy[mask])
    thermal = np.sum(thermal_energy[mask])
    power = np.sum(dissipation_power[mask])
    # .to("dimensionless") enforces the check: if any unit leaks through (a dimensional
    # bug) it raises instead of silently returning a "fraction" with leftover units.
    thermal_over_kinetic = (thermal / kinetic).to("dimensionless")
    rate_fraction = ((power / kinetic) * crossing_time).to("dimensionless")
    print(f"{tag}")
    print(f"   internal/kinetic  (paper's quantity)    = {float(thermal_over_kinetic):.3e}")
    print(f"   (Ediss*V/KE)*(L/v)  [rate construction] = {float(rate_fraction):.3e}")


t_over_tfb, kinetic_energy, thermal_energy, dissipation_power, azimuth = load_star_fields(44)
beam_mask = peak_kinetic_beam(azimuth, kinetic_energy)
print(f"snap_44,  t/t_fb = {t_over_tfb:.3f},  L = Rp,  crossing_time = {crossing_time.to('s'):.1f}\n")
report("paper selection: peak-KE angular beam +/-4.5 deg",
       kinetic_energy, thermal_energy, dissipation_power, beam_mask)
print()
report("all stellar debris (no angular cut)",
       kinetic_energy, thermal_energy, dissipation_power, np.ones(len(kinetic_energy), bool))

snap_44,  t/t_fb = 0.305,  L = Rp,  crossing_time = 516.5 s

paper selection: peak-KE angular beam +/-4.5 deg
   internal/kinetic  (paper's quantity)    = 2.030e-04
   (Ediss*V/KE)*(L/v)  [rate construction] = 4.884e-09



all stellar debris (no angular cut)
   internal/kinetic  (paper's quantity)    = 2.355e-04
   (Ediss*V/KE)*(L/v)  [rate construction] = 2.370e-08


## Comparison

The paper measures at **t = 14.4 days** for their 10^6 Msun BH. For that system (Rp = 100 Rsun,
solar star, beta=1) the fallback time is `t_fb = 40.9 day`, so 14.4 days = **0.35 t_fb**. Our
snap_44 is at 0.305 t_fb, so we're comparing at essentially the **same epoch in fallback time**,
even though the BH masses differ by 100x (our t_fb = 2.58 days vs their 40.9 days).

At that epoch, in the BH frame, with the paper's peak-KE angular-beam selection:

| quantity | ours (0.3 t_fb) | Ryu+ 2025 (0.35 t_fb) |
|---|---|---|
| **internal/kinetic** (their measured quantity) | **~2e-4** | ~4e-5 (converged), ~1e-4 (estimate) |
| dissipation-rate construction `(Ediss*V/KE)*(L/v)` | ~5e-9 | — |

The internal/kinetic ratio is the apples-to-apples comparison, and it's robust (beam selection and
whole-debris sum both give ~2e-4). That's **~2x their 1e-4 estimate and ~5x their converged 4e-5**
— slightly HIGHER, but the same order of magnitude across a 100x difference in BH mass. Good outcome
for a back-of-envelope.

Why the two of our own numbers differ by ~5 orders of magnitude: they measure different things. The
current dissipation rate times one crossing time deposits `Pdiss * crossing_time ~ 2e41 erg`,
whereas the thermal energy already in the gas is `~7e45 erg` — ~10^4x larger. So almost all of the
thermal content was deposited **earlier** (the initial pericenter compression at t~0), and only a
little is being dissipated right now at 0.3 t_fb. The internal/kinetic ratio carries that
accumulated heat (matching the paper); the instantaneous-rate construction only sees the weak
ongoing dissipation.